# OLIVES Metadata Recovery v4

**No GPU needed.** Runtime → Change runtime type → CPU is fine.

### What this does
Re-streams the OLIVES HuggingFace dataset to recover per-embedding metadata.
`File_Path` does not exist in the HuggingFace dataset, so visit ordering uses
`(Eye_ID, BCVA, CST)` as the join key to `Clinical_Data_Images.xlsx` instead.

### Run order
1. Cell 0 — Setup (install, mount Drive)
2. Cell 1 — Stream all metadata, confirm available fields
3. Cell 2 — Save BCVA, CST, Scan (n/49), Patient_ID per embedding row
4. Cell 3 — Verify Eye_ID order matches v3 embeddings

In [ ]:
# ── Cell 0: Setup ─────────────────────────────────────────────────────────
# No GPU needed for this notebook.
!pip install datasets -q

import numpy as np
from google.colab import drive

drive.mount('/content/drive')
OUT_DIR = '/content/drive/MyDrive/synapse/embeddings'
print(f'Drive mounted. Working in: {OUT_DIR}')

In [ ]:
# ── Cell 1: Stream OLIVES metadata ────────────────────────────────────────
# Streams the same dataset/split/config used in v3 encoding.
# Collects every non-Image field so we have the full metadata per row.
# HuggingFace streaming order is deterministic (Parquet shard order),
# so row i here == row i in olives_retfound.npy.

from datasets import load_dataset

print('Streaming OLIVES metadata from HuggingFace (no image decoding)...')
ds = load_dataset(
    'gOLIVES/OLIVES_Dataset',
    'disease_classification',
    split='train',
    streaming=True
)

file_paths   = []
eye_ids_meta = []
labels_meta  = []
all_keys     = None
extra_fields = {}  # catch any other useful columns

for n, sample in enumerate(ds):
    if all_keys is None:
        all_keys = [k for k in sample.keys() if k != 'Image']
        print(f'Available fields (excluding Image): {all_keys}')
        for k in all_keys:
            if k not in ('Eye_ID', 'Disease Label', 'File_Path',
                         'file_path', 'eye_id', 'disease_label'):
                extra_fields[k] = []

    # File_Path — try both capitalizations
    fp = sample.get('File_Path') or sample.get('file_path') or ''
    file_paths.append(str(fp))

    eye_ids_meta.append(sample.get('Eye_ID', -1))
    labels_meta.append(sample.get('Disease Label', -1))

    for k in extra_fields:
        extra_fields[k].append(sample.get(k))

    if (n + 1) % 5000 == 0:
        print(f'  {n + 1} samples...')

print(f'\nDone. Total samples: {len(file_paths)}')

# Save
np.save(f'{OUT_DIR}/olives_filenames.npy',  np.array(file_paths,   dtype=object))
np.save(f'{OUT_DIR}/olives_eye_ids_v4.npy', np.array(eye_ids_meta, dtype=object))
np.save(f'{OUT_DIR}/olives_labels_v4.npy',  np.array(labels_meta,  dtype=float))

print(f'Saved olives_filenames.npy')
print(f'Saved olives_eye_ids_v4.npy  (for cross-check)')
print(f'Saved olives_labels_v4.npy   (for cross-check)')
print(f'\nSample File_Path values:')
for p in file_paths[:5]:
    print(f'  {p!r}')

In [ ]:
# ── Cell 2: Save BCVA, CST, Scan, Patient_ID ──────────────────────────────
# File_Path does not exist in the HuggingFace dataset.
# Instead we join embeddings to visits via (Eye_ID, BCVA, CST).
# This cell saves those fields per embedding row.
#
# If Cell 1 still in memory: uses extra_fields directly (instant).
# If session restarted: re-streams (~10 min, no GPU needed).

import numpy as np

try:
    assert 'extra_fields' in dir() and len(extra_fields.get('BCVA', [])) == 78822
    bcva_arr    = np.array(extra_fields['BCVA'],        dtype=object)
    cst_arr     = np.array(extra_fields['CST'],         dtype=object)
    scan_arr    = np.array(extra_fields['Scan (n/49)'], dtype=object)
    patient_arr = np.array(extra_fields['Patient_ID'],  dtype=object)
    print(f'Using in-memory data from Cell 1: {len(bcva_arr):,} rows')
except (NameError, AssertionError, KeyError):
    print('Re-streaming (session restarted or Cell 1 not run)...')
    from datasets import load_dataset
    ds = load_dataset('gOLIVES/OLIVES_Dataset', 'disease_classification',
                      split='train', streaming=True)
    bcva_list, cst_list, scan_list, patient_list = [], [], [], []
    for n, sample in enumerate(ds):
        bcva_list.append(sample.get('BCVA'))
        cst_list.append(sample.get('CST'))
        scan_list.append(sample.get('Scan (n/49)'))
        patient_list.append(sample.get('Patient_ID'))
        if (n + 1) % 5000 == 0: print(f'  {n + 1} samples...')
    bcva_arr    = np.array(bcva_list,    dtype=object)
    cst_arr     = np.array(cst_list,     dtype=object)
    scan_arr    = np.array(scan_list,    dtype=object)
    patient_arr = np.array(patient_list, dtype=object)
    print(f'Streamed: {len(bcva_arr):,} rows')

np.save(f'{OUT_DIR}/olives_bcva.npy',    bcva_arr)
np.save(f'{OUT_DIR}/olives_cst.npy',     cst_arr)
np.save(f'{OUT_DIR}/olives_scan.npy',    scan_arr)
np.save(f'{OUT_DIR}/olives_patient.npy', patient_arr)

print('Saved: olives_bcva.npy, olives_cst.npy, olives_scan.npy, olives_patient.npy')
print(f'BCVA sample : {bcva_arr[:5]}')
print(f'CST  sample : {cst_arr[:5]}')
print(f'Scan sample : {scan_arr[:5]}')
print(f'Rows total  : {len(bcva_arr):,}')
print('\n--- REPORT THIS OUTPUT BACK ---')
print(f'rows: {len(bcva_arr)}, bcva[:5]: {list(bcva_arr[:5])}, cst[:5]: {list(cst_arr[:5])}, scan[:5]: {list(scan_arr[:5])}')

In [ ]:
# ── Cell 2: Verify ────────────────────────────────────────────────────────
# Cross-checks the re-streamed Eye_IDs against the v3 embedding Eye_IDs.
# If mismatches == 0, the streaming order is identical and filenames are valid.
# If mismatches > 0, something changed in the HuggingFace dataset — report back.

eye_ids_v3 = np.load(f'{OUT_DIR}/olives_eye_ids.npy',    allow_pickle=True)
eye_ids_v4 = np.load(f'{OUT_DIR}/olives_eye_ids_v4.npy', allow_pickle=True)
filenames   = np.load(f'{OUT_DIR}/olives_filenames.npy',  allow_pickle=True)

print(f'v3 embedding rows : {len(eye_ids_v3):,}')
print(f'v4 metadata rows  : {len(eye_ids_v4):,}')

min_len = min(len(eye_ids_v3), len(eye_ids_v4))
mismatches = int(sum(
    str(a) != str(b)
    for a, b in zip(eye_ids_v3[:min_len], eye_ids_v4[:min_len])
))
print(f'Eye_ID mismatches (first {min_len:,} rows): {mismatches}')

empty_paths = sum(1 for p in filenames if not p or p == 'None')
print(f'Empty/None File_Paths: {empty_paths:,} / {len(filenames):,}')

import re
has_visit = sum(1 for p in filenames if re.search(r'/(V|W)\d+/', str(p)))
print(f'Paths with visit marker (/V# or /W#): {has_visit:,} / {len(filenames):,}')

print('\n--- REPORT THIS OUTPUT BACK ---')
print(f'v3 rows: {len(eye_ids_v3)}, v4 rows: {len(eye_ids_v4)}, '
      f'mismatches: {mismatches}, empty paths: {empty_paths}, '
      f'paths with visit: {has_visit}')
print('Sample paths:')
for p in filenames[:5]:
    print(f'  {p!r}')